# Mutual Fund Analysis Dashboard

**Author:** Bakki Tharun Ram Patel  
**Project:** Identifying Top 30 High-Return, Low-Risk Mutual Funds  
**Tools:** Python, Pandas, Scikit-learn, Power BI

---
## Objective
To analyze over 2500 mutual fund schemes and identify top-performing funds with optimal balance between returns and risk using data-driven techniques.

---
## Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load dataset
df = pd.read_csv('Mutual_Fund.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## Data Overview

### Statistical Summary

In [2]:
print("=== Statistical Summary ===")
print(df.describe())

print("\n=== Unique Fund Categories ===")
print(df['category'].unique())

print("\n=== Missing Values ===")
print(df.isnull().sum())

## Data Cleaning

Handle missing values in return columns by filling with mean values.

In [3]:
# Fill missing values with column means
df['returns_3yr'].fillna(df['returns_3yr'].mean(), inplace=True)
df['returns_5yr'].fillna(df['returns_5yr'].mean(), inplace=True)

# Convert columns to numeric
numeric_cols = ['expense_ratio', 'returns_1yr', 'returns_3yr', 'returns_5yr', 
                'sharpe', 'sortino', 'alpha', 'beta']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Missing values after cleaning:")
print(df.isnull().sum())

## Data Normalization

Apply MinMaxScaler to normalize numeric features for fair comparison.

In [4]:
# Initialize scaler
scaler = MinMaxScaler()

# Normalize selected columns
df_normalized = pd.DataFrame(scaler.fit_transform(df[numeric_cols]), 
                              columns=numeric_cols)

# Invert metrics where lower is better (expense ratio and beta)
df_normalized['expense_ratio'] = 1 - df_normalized['expense_ratio']
df_normalized['beta'] = 1 - df_normalized['beta']

print("Normalization completed successfully.")

## Composite Scoring

Create weighted composite score to rank funds based on performance metrics.

In [5]:
# Define weights for each metric
weights = {
    'expense_ratio': 0.20,  # Lower expense ratio is better
    'returns_1yr': 0.15,
    'returns_3yr': 0.20,
    'returns_5yr': 0.15,
    'sharpe': 0.10,
    'sortino': 0.10,
    'alpha': 0.05,
    'beta': 0.05      # Lower beta (less volatility) is better
}

# Calculate composite score
df_normalized['composite_score'] = sum(
    df_normalized[col] * weight for col, weight in weights.items()
)

# Add score back to original dataframe
df['composite_score'] = df_normalized['composite_score']

print("Composite scoring completed.")
print(f"Score range: {df['composite_score'].min():.3f} - {df['composite_score'].max():.3f}")

## Ranking Funds

Rank funds based on composite score and extract top performers.

In [6]:
# Add rank column
df['rank'] = df['composite_score'].rank(ascending=False)

# Sort by rank
df_sorted = df.sort_values(by='rank').reset_index(drop=True)

print("=== Top 5 Ranked Funds ===")
df_sorted[['scheme_name', 'category', 'returns_3yr', 'expense_ratio', 
           'composite_score', 'rank']].head()

## Export Results

Save top 30 mutual funds to Excel file for Power BI dashboard.

In [7]:
# Extract top 30 funds
df_top_30 = df_sorted.head(30)

# Export to Excel
df_top_30.to_excel('top_30_mutual_funds.xlsx', index=False)

print("✅ Exported top 30 mutual funds to 'top_30_mutual_funds.xlsx'")
print(f"\n📊 Top 30 Funds Summary:")
print(f"   - Average 3-Year Return: {df_top_30['returns_3yr'].mean():.2f}%")
print(f"   - Average Expense Ratio: {df_top_30['expense_ratio'].mean():.2f}%")
print(f"   - Fund Categories: {df_top_30['category'].nunique()} unique types")
print(f"   - Top Category: {df_top_30['category'].mode()[0]}")

## Key Insights

1. **Top Performing Category:** Equity funds dominate with highest 3-year returns
2. **Cost Efficiency:** Index funds offer lowest expense ratios
3. **Fund Manager Impact:** Certain managers consistently deliver better risk-adjusted returns
4. **SIP vs Lumpsum:** Average minimum SIP is ₹528, making mutual funds accessible to retail investors

---
## Next Steps

- Import filtered data into Power BI for visualization
- Build interactive dashboard with slicers and KPIs
- Create insights on fund performance by category and AMC

---
## Author

**Bakki Tharun Ram Patel**  
📧 tharunpatel9492@gmail.com  
🔗 [GitHub](https://github.com/Tharunpatel19) | [LinkedIn](https://www.linkedin.com/in/tharun-patel-a68441355/)

*This analysis is part of the Mutual Fund Analysis Dashboard project.*